# ParkCast Vision — Week 2: Cross-Domain Generalization

**핵심 질문**: Week 1의 mAP 0.99는 진짜인가?

Roboflow의 random split은 같은 카메라 시점 사진이 train/test 양쪽에 들어가 있어 data leakage가 의심됨. 진짜 일반화 성능을 측정하기 위해 두 가지 split을 추가로 평가:

1. **Date split**: 시간 기준 분리 (학습 기간 vs 테스트 기간)
2. **Lot split (cross-domain)**: 다른 주차장으로 일반화 — 단, 파일명에 lot 정보가 없으므로 **이미지 임베딩 클러스터링으로 주차장을 자동 발견**

**기대 결과**: Random (0.99) > Date (0.85~0.95) > Lot (0.50~0.75) 의 격차로 도메인 갭이 정량 입증됨.

**산출물**:
- 3가지 split 비교 막대그래프
- t-SNE/UMAP으로 본 도메인 클러스터
- Cross-lot 학습된 모델 (`yolov8n_pklot_lot0.pt`)
- 도메인별 mAP 표

## 1. 환경 + 데이터 준비

In [ ]:
import torch, os, json, glob, random, shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm

print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')

In [ ]:
!pip install -q ultralytics scikit-learn umap-learn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/ParkCast'
MODEL_DIR = f'{PROJECT_ROOT}/models'
RESULTS_DIR = f'{PROJECT_ROOT}/results'
WEEK2_DIR = f'{RESULTS_DIR}/week2'
os.makedirs(WEEK2_DIR, exist_ok=True)

# 데이터 복구 (세션 끊겼을 경우)
DATA_ROOT = '/content'
if not os.path.exists(f'{DATA_ROOT}/train/_annotations.coco.json'):
    print('데이터 복구 중...')
    backup = f'{PROJECT_ROOT}/pklot-dataset.zip'
    assert os.path.exists(backup), f'백업 zip이 {backup}에 없음 — Week 1 노트북 다시 돌려서 데이터 받기'
    !cp "{backup}" /content/
    !unzip -q /content/pklot-dataset.zip -d /content/
    print('복구 완료')

# Week 1에서 만든 YOLO format 데이터도 복구
YOLO_ROOT = '/content/pklot_yolo'
if not os.path.exists(f'{YOLO_ROOT}/train/images'):
    print('YOLO format 재생성 필요 — Week 1 노트북의 섹션 3 (COCO→YOLO 변환) 셀을 실행해주세요')
else:
    print(f'YOLO format OK')

print(f'Train images: {len(glob.glob(f"{DATA_ROOT}/train/*.jpg"))}')
print(f'Valid images: {len(glob.glob(f"{DATA_ROOT}/valid/*.jpg"))}')
print(f'Test images:  {len(glob.glob(f"{DATA_ROOT}/test/*.jpg"))}')

## 2. 전체 이미지 메타데이터 통합

Week 2에서는 split을 새로 짜기 위해 train/valid/test 모든 이미지를 한 풀에 모아둡니다. 파일명에서 날짜·시간 추출.

In [ ]:
import re

# Roboflow 파일명 패턴: '2013-03-22_12_55_08_jpg.rf.해시.jpg'
DATE_PATTERN = re.compile(r'(\d{4}-\d{2}-\d{2})_(\d{2})_(\d{2})_(\d{2})')

def parse_filename(fname):
    m = DATE_PATTERN.search(fname)
    if not m: return None
    date, hh, mm, ss = m.groups()
    return {
        'date': date,
        'hour': int(hh),
        'time': f'{hh}:{mm}:{ss}',
        'datetime': f'{date} {hh}:{mm}:{ss}',
    }

rows = []
for split in ['train', 'valid', 'test']:
    for f in glob.glob(f'{DATA_ROOT}/{split}/*.jpg'):
        fname = os.path.basename(f)
        parsed = parse_filename(fname)
        if parsed:
            rows.append({
                'fname': fname,
                'path': f,
                'orig_split': split,
                **parsed,
            })

meta = pd.DataFrame(rows)
meta['date'] = pd.to_datetime(meta['date'])
print(f'Total images: {len(meta):,}')
print(f'Date range:   {meta["date"].min().date()} ~ {meta["date"].max().date()}')
print(f'\nOrig split distribution:')
print(meta['orig_split'].value_counts())
meta.head()

In [ ]:
# 날짜 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

meta.groupby(meta['date'].dt.to_period('M')).size().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Images per month')
axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=45)

meta['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Images per hour of day')
axes[1].set_xlabel('hour')

plt.tight_layout()
plt.savefig(f'{WEEK2_DIR}/date_distribution.png', dpi=120)
plt.show()

## 3. 이미지 임베딩 추출 (도메인 자동 발견의 출발점)

ImageNet pretrained ResNet50의 backbone으로 모든 이미지를 2048-dim 벡터로 변환. 같은 카메라 시점은 임베딩 공간에서 가까이 모일 것.

In [ ]:
import torchvision.models as models
import torchvision.transforms as T
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ResNet50 backbone (avgpool 직전까지 → 2048-dim)
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Identity()
resnet = resnet.to(device).eval()

tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ImageOnlyDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), idx

ds = ImageOnlyDataset(meta['path'].tolist(), tf)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

embeddings = np.zeros((len(meta), 2048), dtype=np.float32)
with torch.no_grad():
    for x, idx in tqdm(loader, desc='extract embeddings'):
        x = x.to(device)
        feat = resnet(x).cpu().numpy()
        embeddings[idx.numpy()] = feat

print(f'Embeddings: {embeddings.shape}')
np.save(f'{WEEK2_DIR}/embeddings.npy', embeddings)

## 4. 차원축소 + 시각화 — 클러스터가 보이나?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

# PCA로 50차원으로 줄이고 → UMAP으로 2차원
X = StandardScaler().fit_transform(embeddings)
X_pca = PCA(n_components=50, random_state=0).fit_transform(X)
X_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X_pca)

meta['umap_x'] = X_umap[:, 0]
meta['umap_y'] = X_umap[:, 1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(meta['umap_x'], meta['umap_y'], s=4, alpha=0.4, c='steelblue')
ax.set_title('UMAP of ResNet50 embeddings — 클러스터 몇 개가 보이는가?')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
plt.savefig(f'{WEEK2_DIR}/umap_unlabeled.png', dpi=120)
plt.show()

print('보통 PKLot은 3개 주차장(PUCPR, UFPR04, UFPR05)이라 3개 큰 덩어리가 보일 것')
print('덩어리 안에 작은 sub-cluster가 있다면 = 같은 주차장의 다른 날씨')

## 5. K-Means 클러스터링으로 주차장 자동 라벨링

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# k=2~6으로 silhouette score 비교 → 최적 k 찾기
scores = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X_pca)
    s = silhouette_score(X_pca, km.labels_, sample_size=2000, random_state=0)
    scores.append((k, s))
    print(f'  k={k}: silhouette={s:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ks, ss = zip(*scores)
ax.plot(ks, ss, 'o-')
ax.set_xlabel('k (clusters)'); ax.set_ylabel('silhouette score')
ax.set_title('Optimal cluster count')
ax.grid(alpha=0.3)
plt.savefig(f'{WEEK2_DIR}/silhouette.png', dpi=120)
plt.show()

# PKLot은 3개 주차장이라는 사전 지식이 있으므로 k=3 사용
K = 3
km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X_pca)
meta['lot_cluster'] = km.labels_
print(f'\n클러스터별 이미지 수:')
print(meta['lot_cluster'].value_counts().sort_index())

In [ ]:
# 클러스터를 색칠한 UMAP
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e63946', '#2a9d8f', '#f4a261']
for c in range(K):
    sub = meta[meta['lot_cluster'] == c]
    ax.scatter(sub['umap_x'], sub['umap_y'], s=5, alpha=0.5, c=colors[c], label=f'Cluster {c} (n={len(sub):,})')
ax.legend(markerscale=3)
ax.set_title(f'UMAP colored by K-Means (k={K}) — automatically discovered lots')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
plt.savefig(f'{WEEK2_DIR}/umap_clustered.png', dpi=120)
plt.show()

In [ ]:
# 각 클러스터의 샘플 이미지 시각화 — 진짜 다른 주차장으로 묶였는지 눈으로 확인
import cv2

fig, axes = plt.subplots(K, 6, figsize=(20, 3.5*K))
for c in range(K):
    sub = meta[meta['lot_cluster'] == c].sample(min(6, len(meta[meta['lot_cluster']==c])), random_state=42)
    for j, (_, row) in enumerate(sub.iterrows()):
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[c][j].imshow(img)
        axes[c][j].axis('off')
        if j == 0:
            axes[c][j].set_title(f'Cluster {c}\n(n={len(meta[meta.lot_cluster==c]):,})', 
                                  loc='left', fontsize=12, fontweight='bold')
plt.suptitle('Cluster별 샘플 — 진짜 다른 주차장으로 묶였는가?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{WEEK2_DIR}/cluster_samples.png', dpi=120, bbox_inches='tight')
plt.show()

print('체크포인트:')
print('- 각 cluster 내부 6장의 카메라 시점이 일관된가?')
print('- cluster 사이에 시점이 명확히 다른가?')
print('- YES면 클러스터링 성공 → cross-lot split으로 진행')

## 6. 3가지 Split 만들기

이제 같은 데이터로 3가지 split을 준비하고 각각 mAP를 측정합니다.

In [ ]:
# Split A: Random (Week 1과 동일) — 이미 결과 있음
RANDOM_RESULT = {'mAP50': 0.9944, 'mAP50_95': 0.9886, 'precision': 0.9977, 'recall': 0.9975}
print(f'Split A (Random) — Week 1 결과: {RANDOM_RESULT}')

# Split B: Date split — 학습 80% 기간 / 검증 10% / 테스트 10% (시간순)
dates_sorted = sorted(meta['date'].unique())
n_train = int(len(dates_sorted) * 0.8)
n_val = int(len(dates_sorted) * 0.1)
train_dates = set(pd.Timestamp(d) for d in dates_sorted[:n_train])
val_dates = set(pd.Timestamp(d) for d in dates_sorted[n_train:n_train+n_val])
test_dates = set(pd.Timestamp(d) for d in dates_sorted[n_train+n_val:])

def date_split_assign(d):
    if d in train_dates: return 'train'
    if d in val_dates:   return 'val'
    return 'test'

meta['split_date'] = meta['date'].apply(date_split_assign)
print(f'\nSplit B (Date):')
print(meta['split_date'].value_counts())

# Split C: Lot split (cross-domain) — Cluster 0 학습 / Cluster 1 검증 / Cluster 2 테스트
# 각 cluster가 어느 정도 크기인지에 따라 할당이 달라질 수 있어요
cluster_counts = meta['lot_cluster'].value_counts().sort_index()
print(f'\nCluster 크기 (재확인): {cluster_counts.to_dict()}')
# 가장 큰 cluster를 train, 나머지 둘을 val/test로
sorted_clusters = cluster_counts.sort_values(ascending=False).index.tolist()
train_cluster = sorted_clusters[0]
val_cluster = sorted_clusters[1]
test_cluster = sorted_clusters[2]
print(f'  Train cluster: {train_cluster}')
print(f'  Val cluster:   {val_cluster}')
print(f'  Test cluster:  {test_cluster}')

def lot_split_assign(c):
    if c == train_cluster: return 'train'
    if c == val_cluster:   return 'val'
    return 'test'

meta['split_lot'] = meta['lot_cluster'].apply(lot_split_assign)
print(f'\nSplit C (Lot - cross-domain):')
print(meta['split_lot'].value_counts())

## 7. YOLO 데이터 폴더 새로 구성

Week 1에서 만든 `/content/pklot_yolo`는 random split이라 그대로 못 씀. Date split, Lot split 각각 새 폴더 만들기.

In [ ]:
# Week 1에서 만든 라벨을 재활용 (이미지 박스 라벨은 split이 어떻게 바뀌든 동일)
# 각 이미지의 라벨 파일 경로를 찾아두기
label_lookup = {}
for split in ['train', 'valid', 'test']:
    for lbl in glob.glob(f'{YOLO_ROOT}/{split}/labels/*.txt'):
        stem = Path(lbl).stem
        label_lookup[stem] = lbl
print(f'기존 라벨 파일 수: {len(label_lookup):,}')

def build_yolo_split(meta_df, split_col, out_root):
    """meta_df의 split_col 컬럼 기준으로 YOLO 폴더 구성"""
    out_root = Path(out_root)
    for s in ['train', 'val', 'test']:
        (out_root / s / 'images').mkdir(parents=True, exist_ok=True)
        (out_root / s / 'labels').mkdir(parents=True, exist_ok=True)
    
    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc=f'building {out_root.name}'):
        split = row[split_col]
        stem = Path(row['fname']).stem
        
        # 이미지 symlink
        img_dst = out_root / split / 'images' / row['fname']
        if not img_dst.exists():
            try: os.symlink(os.path.abspath(row['path']), img_dst)
            except FileExistsError: pass
        
        # 라벨 symlink (Week 1 라벨 재사용)
        if stem in label_lookup:
            lbl_dst = out_root / split / 'labels' / (stem + '.txt')
            if not lbl_dst.exists():
                try: os.symlink(os.path.abspath(label_lookup[stem]), lbl_dst)
                except FileExistsError: pass
    
    yaml_str = f'''path: {out_root}
train: train/images
val: val/images
test: test/images
nc: 3
names: ['spaces', 'space-empty', 'space-occupied']
'''
    with open(out_root / 'data.yaml', 'w') as f:
        f.write(yaml_str)
    return str(out_root / 'data.yaml')

DATE_ROOT = '/content/pklot_date_split'
LOT_ROOT = '/content/pklot_lot_split'

date_yaml = build_yolo_split(meta, 'split_date', DATE_ROOT)
lot_yaml = build_yolo_split(meta, 'split_lot', LOT_ROOT)

print(f'\nDate split yaml: {date_yaml}')
print(f'Lot split yaml:  {lot_yaml}')

## 8. Date Split 학습 + 평가

각 split당 약 30~45분.

In [ ]:
from ultralytics import YOLO

model_date = YOLO('yolov8n.pt')
model_date.train(
    data=date_yaml, epochs=30, imgsz=640, batch=32, device=0,
    project=WEEK2_DIR, name='split_date',
    patience=10, cos_lr=True, lr0=0.01,
    degrees=2.0, translate=0.05, scale=0.1, fliplr=0.0, mosaic=0.5,
    plots=True,
)

In [ ]:
best_date = YOLO(f'{WEEK2_DIR}/split_date/weights/best.pt')
date_test = best_date.val(
    data=date_yaml, split='test', imgsz=640, batch=32,
    project=WEEK2_DIR, name='split_date_test', plots=True,
)
DATE_RESULT = {
    'mAP50': float(date_test.box.map50),
    'mAP50_95': float(date_test.box.map),
    'precision': float(date_test.box.mp),
    'recall': float(date_test.box.mr),
}
print(f'\nSplit B (Date) results: {DATE_RESULT}')
shutil.copy(f'{WEEK2_DIR}/split_date/weights/best.pt', f'{MODEL_DIR}/yolov8n_pklot_date.pt')

## 9. Lot Split (Cross-Domain) 학습 + 평가

여기가 진짜 흥미로운 부분. 한 주차장에서만 학습한 모델이 다른 주차장에서 얼마나 떨어지나.

In [ ]:
model_lot = YOLO('yolov8n.pt')
model_lot.train(
    data=lot_yaml, epochs=30, imgsz=640, batch=32, device=0,
    project=WEEK2_DIR, name='split_lot',
    patience=10, cos_lr=True, lr0=0.01,
    degrees=2.0, translate=0.05, scale=0.1, fliplr=0.0, mosaic=0.5,
    plots=True,
)

In [ ]:
best_lot = YOLO(f'{WEEK2_DIR}/split_lot/weights/best.pt')
lot_test = best_lot.val(
    data=lot_yaml, split='test', imgsz=640, batch=32,
    project=WEEK2_DIR, name='split_lot_test', plots=True,
)
LOT_RESULT = {
    'mAP50': float(lot_test.box.map50),
    'mAP50_95': float(lot_test.box.map),
    'precision': float(lot_test.box.mp),
    'recall': float(lot_test.box.mr),
}
print(f'\nSplit C (Lot, cross-domain) results: {LOT_RESULT}')
shutil.copy(f'{WEEK2_DIR}/split_lot/weights/best.pt', f'{MODEL_DIR}/yolov8n_pklot_lot.pt')

## 10. 결과 비교 — 메인 시각화

Week 2의 핵심 결과물. 면접 자료에 그대로 들어갈 그림.

In [ ]:
compare_df = pd.DataFrame({
    'Split': ['Random\n(Week 1)', 'Date split\n(temporal)', 'Lot split\n(cross-domain)'],
    'mAP50':    [RANDOM_RESULT['mAP50'],    DATE_RESULT['mAP50'],    LOT_RESULT['mAP50']],
    'mAP50-95': [RANDOM_RESULT['mAP50_95'], DATE_RESULT['mAP50_95'], LOT_RESULT['mAP50_95']],
    'Precision': [RANDOM_RESULT['precision'], DATE_RESULT['precision'], LOT_RESULT['precision']],
    'Recall':    [RANDOM_RESULT['recall'],    DATE_RESULT['recall'],    LOT_RESULT['recall']],
})
print(compare_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
w = 0.2
ax.bar(x - 1.5*w, compare_df['mAP50'],    w, label='mAP50',    color='#264653')
ax.bar(x - 0.5*w, compare_df['mAP50-95'], w, label='mAP50-95', color='#2a9d8f')
ax.bar(x + 0.5*w, compare_df['Precision'],w, label='Precision',color='#e9c46a')
ax.bar(x + 1.5*w, compare_df['Recall'],   w, label='Recall',   color='#f4a261')
ax.set_xticks(x)
ax.set_xticklabels(compare_df['Split'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Metric value')
ax.set_title('Cross-domain generalization gap — same model, different splits', fontsize=13, fontweight='bold')
ax.legend(loc='lower left')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(compare_df['mAP50']):
    ax.text(i - 1.5*w, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{WEEK2_DIR}/SPLIT_COMPARISON.png', dpi=140)
plt.show()

compare_df.to_csv(f'{WEEK2_DIR}/split_comparison.csv', index=False)

## 11. Cross-domain 모델의 실패 케이스 분석

Lot split 모델은 본 적 없는 주차장에서 어떻게 실패하는가?

In [ ]:
# 테스트 cluster 이미지 6장에 lot split 모델 vs random split 모델 둘 다 적용해서 비교
import matplotlib.patches as patches

test_images = meta[meta['split_lot'] == 'test'].sample(6, random_state=42)
best_random = YOLO(f'{MODEL_DIR}/yolov8n_pklot_week1_best.pt')

fig, axes = plt.subplots(6, 2, figsize=(16, 30))
for i, (_, row) in enumerate(test_images.iterrows()):
    img = cv2.imread(row['path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 두 모델 예측
    for j, (model_, label) in enumerate([(best_random, 'Random-trained'), (best_lot, 'Lot-trained')]):
        r = model_(row['path'], conf=0.4, verbose=False)[0]
        ax = axes[i][j]
        ax.imshow(img)
        
        cls_ids = r.boxes.cls.cpu().numpy().astype(int)
        boxes = r.boxes.xyxy.cpu().numpy()
        names = ['spaces', 'space-empty', 'space-occupied']
        n_e = sum(1 for c in cls_ids if 'empty' in names[c].lower())
        n_o = sum(1 for c in cls_ids if 'occupied' in names[c].lower())
        
        for (x1, y1, x2, y2), cls in zip(boxes, cls_ids):
            color = 'lime' if 'empty' in names[cls].lower() else 'red'
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.2, edgecolor=color, facecolor='none'))
        
        ax.set_title(f'{label}   |   Empty={n_e}, Occupied={n_o}, Total={n_e+n_o}', fontsize=10)
        ax.axis('off')

plt.suptitle('Test cluster (unseen lot) — Random-trained vs Lot-trained 비교', fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig(f'{WEEK2_DIR}/cross_domain_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('관찰 포인트:')
print('- Random 모델: test cluster 이미지를 학습 중에도 봤을 가능성 → 검출 정확')
print('- Lot 모델: 진짜 처음 보는 주차장 → 박스 위치가 어긋나거나, 누락이 많거나')
print('- 어디서 실패하는지가 곧 도메인 갭의 정체')

## 12. Week 2 마무리 — 요약 텍스트

In [ ]:
summary = f'''ParkCast Vision - Week 2 (Cross-Domain Generalization)
============================================================

Method:
  1) ResNet50 ImageNet pretrained backbone으로 모든 이미지 임베딩
  2) UMAP + K-Means(k=3) → 주차장(lot) 자동 발견 (라벨 없음)
  3) 같은 데이터로 3가지 split 비교: Random / Date / Lot

Cluster discovery:
  Total: {len(meta):,} images
  Cluster 0: {(meta.lot_cluster==0).sum():,}
  Cluster 1: {(meta.lot_cluster==1).sum():,}
  Cluster 2: {(meta.lot_cluster==2).sum():,}

Results (test mAP50):
  Random split:  {RANDOM_RESULT["mAP50"]:.4f}  (Week 1)
  Date split:    {DATE_RESULT["mAP50"]:.4f}
  Lot split:     {LOT_RESULT["mAP50"]:.4f}

Generalization gap:
  Random → Date: {(RANDOM_RESULT["mAP50"] - DATE_RESULT["mAP50"])*100:+.2f} %p
  Random → Lot:  {(RANDOM_RESULT["mAP50"] - LOT_RESULT["mAP50"])*100:+.2f} %p
  → Random split의 0.99 mAP는 같은 시점 사진의 leakage가 부풀린 결과
  → 진짜 일반화 성능은 Lot split의 {LOT_RESULT["mAP50"]:.2f}로 봐야 함

Models saved:
  {MODEL_DIR}/yolov8n_pklot_date.pt
  {MODEL_DIR}/yolov8n_pklot_lot.pt
'''
with open(f'{WEEK2_DIR}/week2_summary.txt', 'w') as f:
    f.write(summary)
print(summary)

## 다음 단계

**Week 3: 2-Stage Pipeline 비교 (PDF의 ParkCast 방식 검증)**
- 현재 (1-stage): YOLO가 직접 empty/occupied 검출
- 비교 (2-stage): COCO YOLO로 차량만 검출 → 칸 polygon과 IoR 매칭
- 어느 쪽이 cross-domain에서 더 robust한가?
- ipynb의 차량 검출 코드가 그대로 살아남

**Week 4: Gradio 데모 + GitHub README + 발표자료**
- 이미지 업로드 → 실시간 점유 시각화 + 점유율
- 모델 선택 토글 (Random vs Lot-trained)로 일반화 차이 직접 시연
- 면접 어필 포인트 정리